# Experiments with FOLIO

In [ ]:
import sys
sys.path.append('../../utils')
import utility
import math
import itertools
import random
import numpy as np
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv
from pydantic import BaseModel
from openai import OpenAI
from openai.lib._parsing._completions import type_to_response_format_param
import json
import re
from z3 import *
from typing import Literal

load_dotenv()
assert os.getenv("OPENAI_API_KEY") is not None, "You must set your OpenAI API key in the .env file"

openai_client = OpenAI()
seeds_considered = [3, 12, 107, 85, 26]

### Translations of FOLIO instances

In [ ]:
folio_dict = utility.make_jsonl_into_list_dictionaries('../../datasets/folio-train-clean.jsonl')
NL_sentences = {}
FOL_formulas = {}
story_id_already_seen = set()
for item in folio_dict:
    story_id = item['story_id']
    if story_id not in story_id_already_seen:
        story_id_already_seen.add(story_id)
        NL_sentences[story_id] = item['premises']
        FOL_formulas[story_id] = item['premises-FOL']
# remove the story 236 because logically problematic (PValue is not used consistently)
NL_sentences.pop(236)
FOL_formulas.pop(236)
# Delete the sentences in the stories that have the Xor symbol ⊕ (it is treated inconsistently in the dataset)
for i in range(2):
    for story_id in FOL_formulas.keys():
        for sentence in FOL_formulas[story_id]:
            if '⊕' in sentence:
                index = FOL_formulas[story_id].index(sentence)
                FOL_formulas[story_id].pop(index)
                NL_sentences[story_id].pop(index)

In [ ]:
FOL_formulas_Stanford = utility.make_txt_into_list('../../datasets/FOLsentence_Stanford.txt')
list_logical = ['∧', '∨', '→', '↔', '⊕', '∀', '∃','¬']
quantifiers = ['∀', '∃']

complexity_Stanford = []
for formula in FOL_formulas_Stanford:
    complexity_Stanford.append(len([i for i in list_logical if i in formula]))

complexity_FOLIO = []
for story_id in FOL_formulas.keys():
    for formula in FOL_formulas[story_id]:
        complexity_FOLIO.append(len([i for i in list_logical if i in formula]))

print(sum(complexity_Stanford)/len(complexity_Stanford))
print(sum(complexity_FOLIO)/len(complexity_FOLIO))

print(3)
count = 0
for formula in FOL_formulas_Stanford:
    if len([i for i in list_logical if i in formula]) >= 3:
        count += 1
print(count/159)

count = 0
for story_id in FOL_formulas.keys():
    for formula in FOL_formulas[story_id]:
        if len([i for i in list_logical if i in formula]) >= 3:
            count += 1
print(count/1565)

print(4)
count = 0
for formula in FOL_formulas_Stanford:
    if len([i for i in list_logical if i in formula]) >= 4:
        count += 1
print(count/159)

count = 0
for story_id in FOL_formulas.keys():
    for formula in FOL_formulas[story_id]:
        if len([i for i in list_logical if i in formula]) >= 4:
            count += 1
print(count/1565)

print(5)
count = 0
for formula in FOL_formulas_Stanford:
    if len([i for i in list_logical if i in formula]) >= 5:
        count += 1
print(count/159)

count = 0
for story_id in FOL_formulas.keys():
    for formula in FOL_formulas[story_id]:
        if len([i for i in list_logical if i in formula]) >= 5:
            count += 1
print(count/1565)

print('Quantifiers')
print(1)
count = 0
for formula in FOL_formulas_Stanford:
    if len([i for i in quantifiers if i in formula]) >= 1:
        count += 1
print(count/159)

count = 0
for story_id in FOL_formulas.keys():
    for formula in FOL_formulas[story_id]:
        if len([i for i in quantifiers if i in formula]) >= 1:
            count += 1
print(count/1565)
print(2)
count = 0
for formula in FOL_formulas_Stanford:
    if len([i for i in quantifiers if i in formula]) >= 2:
        count += 1
print(count/159)

count = 0
for story_id in FOL_formulas.keys():
    for formula in FOL_formulas[story_id]:
        if len([i for i in quantifiers if i in formula]) >= 2:
            count += 1
print(count/1565)
print(3)
count = 0
for formula in FOL_formulas_Stanford:
    if len([i for i in quantifiers if i in formula]) >= 3:
        count += 1
print(count/159)

count = 0
for story_id in FOL_formulas.keys():
    for formula in FOL_formulas[story_id]:
        if len([i for i in quantifiers if i in formula]) >= 3:
            count += 1
print(count/1565)

print(4)
count = 0
for formula in FOL_formulas_Stanford:
    if len([i for i in quantifiers if i in formula]) >= 4:
        count += 1
print(count/159)

count = 0
for story_id in FOL_formulas.keys():
    for formula in FOL_formulas[story_id]:
        if len([i for i in quantifiers if i in formula]) >= 4:
            count += 1
print(count/1565)

In [ ]:
utility.save_pickle(NL_sentences, '../../datasets/ground-truths_FOLIO.pkl')
utility.save_pickle(FOL_formulas, '../../datasets/FOLsentence_FOLIO.pkl')

In [ ]:
# Get the logical modifications for each FOL_formulas
number_logical_modifications = 3
list_all_modifications_FOLIO = {}
story_exception = []

for story_id in FOL_formulas.keys():
    try: 
        list_all_modifications_FOLIO[story_id] = []
        for formula in FOL_formulas[story_id]:
            list_formula_mod = []
            # put the ground truth
            list_formula_mod.append(utility.FOLFormula(formula))
            # put the equivalent formula in the second position
            equiv_formula = utility.FOLFormula(utility.generate_equivalent_formula(formula))
            list_formula_mod.append(equiv_formula)
            # put the logical modifications
            log_mod = utility.modify_formulas('all_logical', [utility.FOLFormula(formula)], k=3)
            if log_mod and log_mod[0]:
                log_mod = list(set(log_mod[0]))
                if isinstance(log_mod, utility.FOLFormula):
                    list_formula_mod.append(log_mod)
                else:
                    list_formula_mod.extend(log_mod)
            # put the negations at the end
            neg_mod = utility.get_negations([utility.FOLFormula(formula)])
            if isinstance(neg_mod[0], utility.FOLFormula):
                list_formula_mod.append(neg_mod[0])
            else:
                list_formula_mod.extend(neg_mod[0])

            list_all_modifications_FOLIO[story_id].append(list_formula_mod)

    except:
        story_exception.append(story_id)

print(story_exception)
# utility.save_pickle(list_all_modifications_FOLIO, 'Requests/list_modifications_r_FOLIO.pkl')

In [ ]:
# Translate the logical modifications
# The code has been designed to be flexible: translation rules can be modified, like in the following 
translation_rules = [
        # Universal quantifier
            {"type": "regex", "pattern": r"∀([a-zA-Z])", "replacements": ["for all \\1,", "for any \\1,"]},
    
        # Existential quantifier
            {"type": "regex", "pattern": r"∃([a-zA-Z])", "replacements": ["there exists \\1 such that", "there is \\1 such that"]},

        # Implication (the premise and the conclusions are between parenthesis)
            {"type": "impl", "pattern": r"\(([^>]+)\)\s*→\s*\(([^>]+)\)", "replacements": ["if \\1, then \\2", "\\1 implies that \\2"]},

        # Equivalence
            {"type": "string", "pattern": "↔", "replacements": ["if and only if ", "is equivalent to "]},

        # Conjunction
            {"type": "string", "pattern": "∧", "replacements": ["and", "moreover"]},
        
        # Disjunction
            {"type": "string", "pattern": "∨", "replacements": ["or"]}]



list_all_modifications_FOLIO = utility.load_pickle('../../perturbations and translations/Ranking/list_modifications_r_FOLIO.pkl')
constant_meaning = utility.load_pickle('../../perturbations and translations/Constant_meaning_FOLIO.pkl')
relational_meaning = utility.load_pickle('../../perturbations and translations/Relational_meaning_FOLIO.pkl')
combination = [0, 1, 0, 1, 0, 1, 0] # In our experiments, we just consider this combination of translation rules, which corresponds to those specified in the paper's Appendix

list_translated = {}
list_exception = []

for story_id in list_all_modifications_FOLIO.keys():
    try: 
        list_translated[story_id] = utility.translate_formulas_FOLIO(list_all_modifications_FOLIO[story_id], constant_meaning[story_id], relational_meaning[story_id]['dict_pos'], relational_meaning[story_id]['dict_neg'], translation_rules, combination)
    except:
        list_exception.append(story_id)

#utility.save_pickle(list_translated, '../../perturbations and translations/Ranking/list_translations_r_FOLIO.pkl')

In [ ]:
print(list_translated[8][2])

In [ ]:
# Shuffle the position and store them for when we will prompt the model

list_new_positions_FOLIO = {}
for story_id in FOL_formulas.keys():
    list_new_positions_FOLIO[story_id] = [[] for i in FOL_formulas[story_id]]
# if list_new_positions_FOLIO[0] = [1, 3, 2], then it means that the sentence list_all_modifications[0][i], in the user prompt is the sentence no. list_new_positions_FOLIO[0][i]. 
    for i in range(len(FOL_formulas[story_id])):
        list_new_positions_FOLIO[story_id][i] = [k+1 for k in range(len(list_all_modifications_FOLIO[story_id][i]))]
        random.shuffle(list_new_positions_FOLIO[story_id][i])

#utility.save_pickle(list_new_positions_FOLIO, '../../perturbations and translations/Ranking/list_shuffled_positions_r_FOLIO.pkl')


In [ ]:
system_prompt_FOL = """
You are an expert evaluator specializing in ranking some FOL statements according to their semantic similarity. You will be given a reference and other statements; your task is to rank these statements depending on whether they convey the same meaning of the reference or not.

Instructions:
- Ignore difference in word order or logical symbol used. You are only interested in the semantic behind the statements. It is possible that two statements use different logical operator but they have the same semantic meaning and this is the only thing you have to focus on;
- If one of the statement is equivalent to the reference, it should be ranked first;
- If one of the statement is equivalent to the negation of the reference, i.e. if it has the opposite meaning of the reference, it should be ranked last.
- If two statements are equivalent, they should be ranked in adjacent positions

Input Format:
Reference: <reference>
Statement1: <sentence1>
Statement2: <sentence2>
...
StatementN : <sentenceN>

Output Format:
Return, after a reasoning stage, the numbers of the statements in order, from the number of the statement that is the most similar to the reference to the one that is the least ([number_of_the_statement_ranked_first, number_of_the_statement_ranked_second, ... number_of_the_statement_ranked_last]).
"""

system_prompt_NL = """
You are an expert evaluator specializing in ranking some NL sentences according to their semantic similarity. You will be given a reference and other sentences; your task is to rank these sentences depending on whether they convey the same meaning of the reference or not.

Instructions:
- Ignore difference in grammar, syntax, style, or word order. You are only interested in the semantic behind the sentences. It is possible that two sentences have a different logical structure but they convey the same logical meaning and this is the only thing you have to focus on;
- If one of the sentences is equivalent to the reference, it should be ranked first;
- If one of the sentences is equivalent to the negation of the reference, i.e. if it has the opposite meaning of the reference, it should be ranked last.
- If two sentences are equivalent, they should be ranked in adjacent positions

Input Format:
Reference: <reference>
Sentence1: <sentence1>
Sentence2: <sentence2>
...
SentenceN : <sentenceN>

Output Format:
Return, after a reasoning stage, the numbers of the sentences in order, from the number of the sentence that is the most similar to the reference to the one that is the least. ([number_of_the_statement_ranked_first, number_of_the_statement_ranked_second, ... number_of_the_statement_ranked_last]).
"""

user_prompt_format = """
Reference: <reference>
<user_prompt_list>
"""

user_prompts_dictionary_NL = {}
user_prompts_dictionary_FOL = {}

# Load the translations
translations = utility.load_pickle('../../perturbations and translations/Ranking/list_translations_r_FOLIO.pkl')
list_new_positions = utility.load_pickle('../../perturbations and translations/Ranking/list_shuffled_positions_r_FOLIO.pkl')


# Prepare the FOL and the NL prompts
for story_id in FOL_formulas.keys():
    for i in range(len(FOL_formulas[story_id])):
        # the key (with which we will refer to the question in the future) will be {story_id}_{number_instance}
        key = f'{story_id}_{i}'

        # FOL prompts
        user_prompt = user_prompt_format.replace('<reference>', NL_sentences[story_id][i])
        sentence_list = ''
        for j, item in enumerate(list_new_positions[story_id][i]):
            sentence_list += f'Statement{j+1}: {list_all_modifications_FOLIO[story_id][i][(list_new_positions[story_id][i]).index(j+1)]}\n'
        user_prompt = user_prompt.replace('<user_prompt_list>', sentence_list)
        user_prompts_dictionary_FOL[key] = user_prompt

        # NL prompts
        user_prompt = user_prompt_format.replace('<reference>', NL_sentences[story_id][i])
        sentence_list = ''
        for j, item in enumerate(list_new_positions[story_id][i]):
            transl = translations[story_id][i][(list_new_positions[story_id][i]).index(j+1)]
            sentence_list += f'Sentence{j+1}: {transl}\n'
        user_prompt = user_prompt.replace('<user_prompt_list>', sentence_list)
        user_prompts_dictionary_NL[key] = user_prompt


In [ ]:
#### CALL API: NOT TO RUN
model = 'gpt-4o-mini'
class response_format(BaseModel):
    reasoning: str
    answer: list[int]

all_requests = []

for seed in seeds_considered:
    # for NL
    custom_id_NL = f'FOLIO_Ranking_NL_{seed}'
    requests_NL = utility.create_openai_list_requests_for_batch_completions(model, custom_id_NL, system_prompt_NL, user_prompts_dictionary_NL, type_to_response_format_param(response_format), seed = seed, max_tokens = 2500)

    # for FOL
    custom_id_FOL = f'FOLIO_Ranking_FOL_{seed}'
    requests_FOL = utility.create_openai_list_requests_for_batch_completions(model, custom_id_FOL, system_prompt_FOL, user_prompts_dictionary_FOL, type_to_response_format_param(response_format), seed = seed, max_tokens = 2500)

    all_requests += requests_NL + requests_FOL

# Save the requests in a json file and call openai 
utility.make_list_dictionaries_into_jsonl(all_requests, f'Requests/FOLIO_ranking_NL_FOL_{model}_seed.jsonl')

# # Call API (Remove comments if you want to call API)
# FOLIO_ranking_NL_FOL_4omini_seed, FOLIO_ranking_NL_FOL_4omini_batch_seed = utility.call_openai_api_batch_completions(all_requests, 'Requests', f'FOLIO_ranking_NL_FOL_{model}_seed.jsonl')

# # save the batch id
# utility.save_pickle(FOLIO_ranking_NL_FOL_4omini_seed, f'Requests/batch_ids/FOLIO_ranking_NL_FOL_{model}_seed.pkl')

In [ ]:
# load the batch
model = 'gpt-4o-mini'
FOLIO_ranking_NL_FOL_4omini_id_seed = utility.load_pickle(f'Requests/batch_ids/FOLIO_ranking_NL_FOL_{model}_seed.pkl')
FOLIO_ranking_NL_FOL_4omini_batch_seed = openai_client.batches.retrieve(FOLIO_ranking_NL_FOL_4omini_id_seed)
print(FOLIO_ranking_NL_FOL_4omini_batch_seed.status)
print(FOLIO_ranking_NL_FOL_4omini_batch_seed)

if utility.extract_batch_errors_openai(FOLIO_ranking_NL_FOL_4omini_batch_seed):
    print('Error file is present.')
else:
    output = utility.extract_batch_outputs_openai(FOLIO_ranking_NL_FOL_4omini_batch_seed)
    if not output:
        print('No output file available')

utility.save_pickle(output, f'Results/FOLIO_r_NL_FOL_{model}_seed.pkl')

In [ ]:
# Clean the results and save them in order to get the clean results and then the performances
model = 'gpt-4o-mini'
output = utility.load_pickle(f'Results/FOLIO_r_NL_FOL_{model}_seed.pkl')

list_responses = {}
list_error_parsing = []

for seed in seeds_considered:
    list_responses[seed] = []

for item in output:
    number_instance = int(item['custom_id'].split('_')[-1]) 
    story_id = int(item['custom_id'].split('_')[-2])
    seed = int(item['custom_id'].split('_')[-3])
    NLorFOL = str(item['custom_id'].split('_')[-4])
    try: 
        message = json.loads(item['response']['body']['choices'][0]['message']['content'])
        answer = list(message['answer'])
        reasoning = message['reasoning']
        list_responses[seed].append({
                'story_id' : story_id,
                'number_instance' : number_instance,
                'answer': answer,
                'NLorFOL' : NLorFOL,
                'reasoning': reasoning
            })

    except:
        list_error_parsing.append(f'{seed}_{NLorFOL}_{story_id}_{number_instance}')

print(f'{list_error_parsing=}')
print(len(list_error_parsing))

utility.save_pickle(list_responses, f'Results/Clean Results/FOLIO_r_NL_FOL_{model}_seed.pkl')


In [ ]:
model = 'gpt-4o-mini'
clean_results = utility.load_pickle( f'Results/Clean Results/FOLIO_r_NL_FOL_{model}_seed.pkl')
list_new_positions = utility.load_pickle('../../perturbations and translations/Ranking/list_shuffled_positions_r_FOLIO.pkl')
list_all_modifications_FOLIO = utility.load_pickle('../../perturbations and translations/Ranking/list_modifications_r_FOLIO.pkl')

gt_and_equivalent_first_2_positions_NL = {}
negations_last_2_positions_NL = {}
all_ranking_correct_NL = {}
gt_and_equivalent_first_2_positions_FOL = {}
negations_last_2_positions_FOL = {}
all_ranking_correct_FOL = {}
list_instances_with_response_NL = {}
list_instances_with_response_FOL = {}

for seed in seeds_considered:
    gt_and_equivalent_first_2_positions_NL[seed] = {}
    negations_last_2_positions_NL[seed] = {}
    all_ranking_correct_NL[seed] = {}
    gt_and_equivalent_first_2_positions_FOL[seed] = {}
    negations_last_2_positions_FOL[seed] = {}
    all_ranking_correct_FOL[seed] = {}
    list_instances_with_response_NL[seed] = {}
    list_instances_with_response_FOL[seed] = {}
    for story_id in FOL_formulas.keys():
        gt_and_equivalent_first_2_positions_NL[seed][story_id] = []
        negations_last_2_positions_NL[seed][story_id] = []
        all_ranking_correct_NL[seed][story_id] = []
        gt_and_equivalent_first_2_positions_FOL[seed][story_id] = []
        negations_last_2_positions_FOL[seed][story_id] = []
        all_ranking_correct_FOL[seed][story_id] = []
        list_instances_with_response_NL[seed][story_id] = []
        list_instances_with_response_FOL[seed][story_id] = []


for seed in clean_results.keys():
    for line in clean_results[seed]:
        story_id, number_instance, answer, NLorFOL = line['story_id'], line['number_instance'], line['answer'], line['NLorFOL'] 
        if NLorFOL == 'NL':
            list_instances_with_response_NL[seed][story_id].append(number_instance)
        else:
            list_instances_with_response_FOL[seed][story_id].append(number_instance)
        # for any formula, the ground truth and its equivalent should always be selected (unless we are comparing them)
        if set(list_new_positions[story_id][number_instance][0:2]) == set(answer[0:2]):
            if NLorFOL == 'NL':
                gt_and_equivalent_first_2_positions_NL[seed][story_id].append(number_instance)
            else:
                gt_and_equivalent_first_2_positions_FOL[seed][story_id].append(number_instance)
        # for any formula, the negation and its equivalent should never be selected (unless we are comparing them)
        last_position = len(list_all_modifications_FOLIO[story_id][number_instance])-1
        forelast_position = last_position-1
        if set(list_new_positions[story_id][number_instance][forelast_position:]) == set(answer[forelast_position:]):
            if NLorFOL == 'NL':
                negations_last_2_positions_NL[seed][story_id].append(number_instance)
            else:
                negations_last_2_positions_FOL[seed][story_id].append(number_instance)


# compute the all ranking_correct
for seed in seeds_considered:
    for story_id in gt_and_equivalent_first_2_positions_NL[seed].keys():
        for item in gt_and_equivalent_first_2_positions_NL[seed][story_id]:
            if item in negations_last_2_positions_NL[seed][story_id]:
                all_ranking_correct_NL[seed][story_id].append(item)

for seed in seeds_considered:
    for story_id in gt_and_equivalent_first_2_positions_FOL[seed].keys():
        for item in gt_and_equivalent_first_2_positions_FOL[seed][story_id]:
            if item in negations_last_2_positions_FOL[seed][story_id]:
                all_ranking_correct_FOL[seed][story_id].append(item)

# Print the results
print('FOLIO')
print('_'*10)
print('NL')
for seed in gt_and_equivalent_first_2_positions_NL.keys():
    gt_and_NL = 0
    for story_id in gt_and_equivalent_first_2_positions_NL[seed].keys():
        gt_and_NL += len(gt_and_equivalent_first_2_positions_NL[seed][story_id])

    gt_and_FOL = 0
    for story_id in gt_and_equivalent_first_2_positions_FOL[seed].keys():
        gt_and_FOL += len(gt_and_equivalent_first_2_positions_FOL[seed][story_id])

    neg_2_NL = 0
    for story_id in negations_last_2_positions_NL[seed].keys():
        neg_2_NL += len(negations_last_2_positions_NL[seed][story_id])

    neg_2_FOL = 0
    for story_id in negations_last_2_positions_FOL[seed].keys():
        neg_2_FOL += len(negations_last_2_positions_FOL[seed][story_id])

    list_with_resp_NL = 0
    for story_id in list_instances_with_response_NL[seed].keys():
        list_with_resp_NL += len(list_instances_with_response_NL[seed][story_id])

    list_with_resp_FOL = 0
    for story_id in list_instances_with_response_FOL[seed].keys():
        list_with_resp_FOL += len(list_instances_with_response_FOL[seed][story_id])

    all_corr_NL = 0
    for story_id in all_ranking_correct_NL[seed].keys():
        all_corr_NL += len(all_ranking_correct_NL[seed][story_id])

    all_corr_FOL = 0
    for story_id in all_ranking_correct_FOL[seed].keys():
        all_corr_FOL += len(all_ranking_correct_FOL[seed][story_id])


    print('Model_considered: ', model)
    print('Seed:', seed)
    print('No. of instances with a parsable answer', list_with_resp_NL)
    print('No. of times in which the ground truth and the equivlent statement in the first two positions: ', gt_and_NL/list_with_resp_NL)
    print('No. of times in which the two equivalent negations are in the last two positions: ', neg_2_NL/list_with_resp_NL)
    print('No. of times in which all the ranking is correct (intersection of the above): ', all_corr_NL/list_with_resp_NL)
    print('_'*10)

    print('FOL')
    print('No. of instances with a parsable answer', list_with_resp_FOL)
    print('No. of times in which the ground truth and the equivlent statement in the first two positions: ', gt_and_FOL/list_with_resp_FOL)
    print('No. of times in which the two equivalent negations are in the last two positions: ', neg_2_FOL/list_with_resp_FOL)
    print('No. of times in which all the ranking is correct (intersection of the above): ', all_corr_FOL/list_with_resp_FOL)
    print('_'*10)
    print('*'*20)

# Do the same with o3 mini

In [ ]:
#### CALL API: NOT TO RUN
model = 'o3-mini'
class response_format(BaseModel):
    reasoning: str
    answer: list[int]

all_requests = []

for seed in seeds_considered:
    # for NL
    custom_id_NL = f'FOLIO_Ranking_NL_{seed}'
    requests_NL = utility.create_openai_list_requests_for_batch_completions(model, custom_id_NL, system_prompt_NL, user_prompts_dictionary_NL, type_to_response_format_param(response_format), seed = seed, max_tokens = 10000)

    # for FOL
    custom_id_FOL = f'FOLIO_Ranking_FOL_{seed}'
    requests_FOL = utility.create_openai_list_requests_for_batch_completions(model, custom_id_FOL, system_prompt_FOL, user_prompts_dictionary_FOL, type_to_response_format_param(response_format), seed = seed, max_tokens = 10000)

    all_requests += requests_NL + requests_FOL

# Save the requests in a json file and call openai 
utility.make_list_dictionaries_into_jsonl(all_requests, f'Requests/FOLIO_ranking_NL_FOL_{model}_seed.jsonl')

# Call API (Remove comments if you want to call the API)
# FOLIO_ranking_NL_FOL_o3mini_seed, FOLIO_ranking_NL_FOL_o3mini_batch_seed = utility.call_openai_api_batch_completions(all_requests, 'Requests', f'FOLIO_ranking_NL_FOL_{model}_seed.jsonl')

# # save the batch id
# utility.save_pickle(FOLIO_ranking_NL_FOL_o3mini_seed, f'Requests/batch_ids/FOLIO_ranking_NL_FOL_{model}_seed.pkl')

In [ ]:
# load the batch
model = 'o3-mini'
FOLIO_ranking_NL_FOL_o3mini_id_seed = utility.load_pickle(f'Requests/batch_ids/FOLIO_ranking_NL_FOL_{model}_seed.pkl')
FOLIO_ranking_NL_FOL_o3mini_batch_seed = openai_client.batches.retrieve(FOLIO_ranking_NL_FOL_o3mini_id_seed)
print(FOLIO_ranking_NL_FOL_o3mini_batch_seed.status)
print(FOLIO_ranking_NL_FOL_o3mini_batch_seed)

if utility.extract_batch_errors_openai(FOLIO_ranking_NL_FOL_o3mini_batch_seed):
    print('Error file is present.')
else:
    output = utility.extract_batch_outputs_openai(FOLIO_ranking_NL_FOL_o3mini_batch_seed)
    if not output:
        print('No output file available')

utility.save_pickle(output, f'Results/FOLIO_r_NL_FOL_{model}_seed.pkl')

In [ ]:
# Clean the results and save them in order to get the clean results and then the performances
model = 'o3-mini'
output = utility.load_pickle(f'Results/FOLIO_r_NL_FOL_{model}_seed.pkl')

list_responses = {}
list_error_parsing = []

for seed in seeds_considered:
    list_responses[seed] = []

for item in output:
    number_instance = int(item['custom_id'].split('_')[-1]) 
    story_id = int(item['custom_id'].split('_')[-2])
    seed = int(item['custom_id'].split('_')[-3])
    NLorFOL = str(item['custom_id'].split('_')[-4])
    try: 
        message = json.loads(item['response']['body']['choices'][0]['message']['content'])
        answer = list(message['answer'])
        reasoning = message['reasoning']
        list_responses[seed].append({
                'story_id' : story_id,
                'number_instance' : number_instance,
                'answer': answer,
                'NLorFOL' : NLorFOL,
                'reasoning': reasoning
            })

    except:
        list_error_parsing.append(f'{seed}_{NLorFOL}_{story_id}_{number_instance}')

print(f'{list_error_parsing=}')
print(len(list_error_parsing))

utility.save_pickle(list_responses, f'Results/Clean Results/FOLIO_r_NL_FOL_{model}_seed.pkl')


In [ ]:
model = 'o3-mini'
clean_results = utility.load_pickle( f'Results/Clean Results/FOLIO_r_NL_FOL_{model}_seed.pkl')
list_new_positions = utility.load_pickle('../../perturbations and translations/Ranking/list_shuffled_positions_r_FOLIO.pkl')
list_all_modifications_FOLIO = utility.load_pickle('../../perturbations and translations/Ranking/list_modifications_r_FOLIO.pkl')

gt_and_equivalent_first_2_positions_NL = {}
negations_last_2_positions_NL = {}
all_ranking_correct_NL = {}
gt_and_equivalent_first_2_positions_FOL = {}
negations_last_2_positions_FOL = {}
all_ranking_correct_FOL = {}
list_instances_with_response_NL = {}
list_instances_with_response_FOL = {}
neg_in_last_two_NL = {}
nnf_in_last_two_NL = {}
neg_in_last_two_FOL = {}
nnf_in_last_two_FOL = {}

for seed in seeds_considered:
    gt_and_equivalent_first_2_positions_NL[seed] = {}
    negations_last_2_positions_NL[seed] = {}
    all_ranking_correct_NL[seed] = {}
    gt_and_equivalent_first_2_positions_FOL[seed] = {}
    negations_last_2_positions_FOL[seed] = {}
    all_ranking_correct_FOL[seed] = {}
    list_instances_with_response_NL[seed] = {}
    list_instances_with_response_FOL[seed] = {}   

    neg_in_last_two_NL[seed] = {}
    nnf_in_last_two_NL[seed] = {}
    neg_in_last_two_FOL[seed] = {}
    nnf_in_last_two_FOL[seed] = {}
    for story_id in FOL_formulas.keys():
        gt_and_equivalent_first_2_positions_NL[seed][story_id] = []
        negations_last_2_positions_NL[seed][story_id] = []
        all_ranking_correct_NL[seed][story_id] = []
        gt_and_equivalent_first_2_positions_FOL[seed][story_id] = []
        negations_last_2_positions_FOL[seed][story_id] = []
        all_ranking_correct_FOL[seed][story_id] = []
        list_instances_with_response_NL[seed][story_id] = []
        list_instances_with_response_FOL[seed][story_id] = []

        neg_in_last_two_NL[seed][story_id] = []
        nnf_in_last_two_NL[seed][story_id] = []
        neg_in_last_two_FOL[seed][story_id] = []
        nnf_in_last_two_FOL[seed][story_id] = []


for seed in clean_results.keys():
    for line in clean_results[seed]:
        story_id, number_instance, answer, NLorFOL = line['story_id'], line['number_instance'], line['answer'], line['NLorFOL'] 
        if NLorFOL == 'NL':
            list_instances_with_response_NL[seed][story_id].append(number_instance)
        else:
            list_instances_with_response_FOL[seed][story_id].append(number_instance)
        # for any formula, the ground truth and its equivalent should always be selected (unless we are comparing them)
        if set(list_new_positions[story_id][number_instance][0:2]) == set(answer[0:2]):
            if NLorFOL == 'NL':
                gt_and_equivalent_first_2_positions_NL[seed][story_id].append(number_instance)
            else:
                gt_and_equivalent_first_2_positions_FOL[seed][story_id].append(number_instance)
        # for any formula, the negation and its equivalent should never be selected (unless we are comparing them)
        last_position = len(list_all_modifications_FOLIO[story_id][number_instance])-1
        forelast_position = last_position-1
        if set(list_new_positions[story_id][number_instance][forelast_position:]) == set(answer[forelast_position:]):
            if NLorFOL == 'NL':
                negations_last_2_positions_NL[seed][story_id].append(number_instance)
            else:
                negations_last_2_positions_FOL[seed][story_id].append(number_instance)

        if list_new_positions[story_id][number_instance][last_position] in set(answer[forelast_position:]):
            if NLorFOL == 'NL':
                nnf_in_last_two_NL[seed][story_id].append(number_instance)
            else:
                nnf_in_last_two_FOL[seed][story_id].append(number_instance)
        if list_new_positions[story_id][number_instance][forelast_position] in set(answer[forelast_position:]):
            if NLorFOL == 'NL':
                neg_in_last_two_NL[seed][story_id].append(number_instance)
            else:
                neg_in_last_two_FOL[seed][story_id].append(number_instance)


# compute the all ranking_correct
for seed in seeds_considered:
    for story_id in gt_and_equivalent_first_2_positions_NL[seed].keys():
        for item in gt_and_equivalent_first_2_positions_NL[seed][story_id]:
            if item in negations_last_2_positions_NL[seed][story_id]:
                all_ranking_correct_NL[seed][story_id].append(item)

for seed in seeds_considered:
    for story_id in gt_and_equivalent_first_2_positions_FOL[seed].keys():
        for item in gt_and_equivalent_first_2_positions_FOL[seed][story_id]:
            if item in negations_last_2_positions_FOL[seed][story_id]:
                all_ranking_correct_FOL[seed][story_id].append(item)

# Print the results
print('FOLIO')
print('_'*10)
print('NL')
for seed in gt_and_equivalent_first_2_positions_NL.keys():
    gt_and_NL = 0
    for story_id in gt_and_equivalent_first_2_positions_NL[seed].keys():
        gt_and_NL += len(gt_and_equivalent_first_2_positions_NL[seed][story_id])

    gt_and_FOL = 0
    for story_id in gt_and_equivalent_first_2_positions_FOL[seed].keys():
        gt_and_FOL += len(gt_and_equivalent_first_2_positions_FOL[seed][story_id])

    neg_2_NL = 0
    for story_id in negations_last_2_positions_NL[seed].keys():
        neg_2_NL += len(negations_last_2_positions_NL[seed][story_id])

    neg_2_FOL = 0
    for story_id in negations_last_2_positions_FOL[seed].keys():
        neg_2_FOL += len(negations_last_2_positions_FOL[seed][story_id])

    list_with_resp_NL = 0
    for story_id in list_instances_with_response_NL[seed].keys():
        list_with_resp_NL += len(list_instances_with_response_NL[seed][story_id])

    list_with_resp_FOL = 0
    for story_id in list_instances_with_response_FOL[seed].keys():
        list_with_resp_FOL += len(list_instances_with_response_FOL[seed][story_id])

    all_corr_NL = 0
    for story_id in all_ranking_correct_NL[seed].keys():
        all_corr_NL += len(all_ranking_correct_NL[seed][story_id])

    all_corr_FOL = 0
    for story_id in all_ranking_correct_FOL[seed].keys():
        all_corr_FOL += len(all_ranking_correct_FOL[seed][story_id])


    neg_in_last_two_NL_count = 0
    for story_id in neg_in_last_two_NL[seed].keys():
        neg_in_last_two_NL_count += len(neg_in_last_two_NL[seed][story_id])
    neg_in_last_two_FOL_count = 0
    for story_id in neg_in_last_two_FOL[seed].keys():
        neg_in_last_two_FOL_count += len(neg_in_last_two_FOL[seed][story_id])
    nnf_in_last_two_NL_count = 0
    for story_id in nnf_in_last_two_NL[seed].keys():
        nnf_in_last_two_NL_count += len(nnf_in_last_two_NL[seed][story_id])
    nnf_in_last_two_FOL_count = 0
    for story_id in nnf_in_last_two_FOL[seed].keys():
        nnf_in_last_two_FOL_count += len(nnf_in_last_two_FOL[seed][story_id])
    check = 0
    for story_id in NL_sentences.keys():
        check += len([i for i in neg_in_last_two_NL[seed][story_id] if i in nnf_in_last_two_NL[seed][story_id]])
    




    print('Model_considered: ', model)
    print('Seed:', seed)
    print('No. of instances with a parsable answer', list_with_resp_NL)
    print('No. of times in which the ground truth and the equivlent statement in the first two positions: ', gt_and_NL/list_with_resp_NL)
    print('No. of times in which the two equivalent negations are in the last two positions: ', neg_2_NL/list_with_resp_NL)
    print('No. of times in which all the ranking is correct (intersection of the above): ', all_corr_NL/list_with_resp_NL)
    print('neg in last two:', neg_in_last_two_NL_count/1565)
    print('nnf in last two:', nnf_in_last_two_NL_count/1565)
    print('_'*10)
    print('FOL')
    print('No. of instances with a parsable answer', list_with_resp_FOL)
    print('No. of times in which the ground truth and the equivlent statement in the first two positions: ', gt_and_FOL/list_with_resp_FOL)
    print('No. of times in which the two equivalent negations are in the last two positions: ', neg_2_FOL/list_with_resp_FOL)
    print('No. of times in which all the ranking is correct (intersection of the above): ', all_corr_FOL/list_with_resp_FOL)
    print('neg in last two:', neg_in_last_two_FOL_count/1565)
    print('nnf in last two:', nnf_in_last_two_FOL_count/1565)
    print('_'*10)
    print('*'*20)